In [1]:
import pandas as pd
import numpy as np
import fits_convert_trackmate
import fits_convert_trackmate_alt
import fits_convert
import msd_calc
import calc_anisotropy
import calc_lifetime
import utils

In [2]:
## Configure parameters

# Directories to use. Formatted as a dictionary with filename: condition. Graphs are ordered by condition's first appearance
# Since everything goes to Matplotlib, condition can be formatted with TEX to get superscripts/special chars
# Fun latex tip: use \\mathsf{} to get non-italic font
# Since names get clunky quickly, I assign the names to variables and use those for configuration


DATA_DIRECTORIES = {
    "/home/dl/work/lsa-jsbiteen/MIGRATED/Lab_Members/Sam_Steen/Data/260115_PAmCherry-Swi6_wt_20ms": "wt",
    "/home/dl/work/lsa-jsbiteen/MIGRATED/Lab_Members/Sam_Steen/Data/260116_PAmCherry-Swi6_wt-and-sj_20ms/wt": "wt",
    "/home/dl/work/lsa-jsbiteen/MIGRATED/Lab_Members/Sam_Steen/Data/260117_PAmCherry-Swi6_sj-and-wt_20ms/wt": "wt",
    "/home/dl/work/lsa-jsbiteen/MIGRATED/Lab_Members/Sam_Steen/Data/260119_PAmCherry-Swi6_sj-and-wt_20ms/wt": "wt",
    # "/home/dl/work/lsa-jsbiteen/MIGRATED/Lab_Members/Sam_Steen/Data/260116_PAmCherry-Swi6_wt-and-sj_20ms/sj": "sj",
    # "/home/dl/work/lsa-jsbiteen/MIGRATED/Lab_Members/Sam_Steen/Data/260117_PAmCherry-Swi6_sj-and-wt_20ms/sj": "sj",
    # "/home/dl/work/lsa-jsbiteen/MIGRATED/Lab_Members/Sam_Steen/Data/260119_PAmCherry-Swi6_sj-and-wt_20ms/sj": "sj",
    "/home/dl/work/lsa-jsbiteen/MIGRATED/Lab_Members/Sam_Steen/Data/260204_PAmCherry-Swi6_varied_20ms/wt": "wt",
    # "/home/dl/work/lsa-jsbiteen/MIGRATED/Lab_Members/Sam_Steen/Data/260204_PAmCherry-Swi6_varied_20ms/chim": "chim",
    # "/home/dl/work/lsa-jsbiteen/MIGRATED/Lab_Members/Sam_Steen/Data/260204_PAmCherry-Swi6_varied_20ms/chim16": "chim16",
    # "/home/dl/work/lsa-jsbiteen/MIGRATED/Lab_Members/Sam_Steen/Data/260205_PAmCherry-Swi6_varied_20ms/sj": "sj",
    # "/home/dl/work/lsa-jsbiteen/MIGRATED/Lab_Members/Sam_Steen/Data/260205_PAmCherry-Swi6_varied_20ms/chim": "chim",
    # "/home/dl/work/lsa-jsbiteen/MIGRATED/Lab_Members/Sam_Steen/Data/260205_PAmCherry-Swi6_varied_20ms/chim16": "chim16",
    "/home/dl/work/lsa-jsbiteen/MIGRATED/Lab_Members/Sam_Steen/Data/260213_PAmCherry-Swi6_varied_20ms/wt": "wt",
    # "/home/dl/work/lsa-jsbiteen/MIGRATED/Lab_Members/Sam_Steen/Data/260213_PAmCherry-Swi6_varied_20ms/sj": "sj",
    # "/home/dl/work/lsa-jsbiteen/MIGRATED/Lab_Members/Sam_Steen/Data/260213_PAmCherry-Swi6_varied_20ms/chim": "chim",
    # "/home/dl/work/lsa-jsbiteen/MIGRATED/Lab_Members/Sam_Steen/Data/260213_PAmCherry-Swi6_varied_20ms/chim16": "chim16",
    # "/home/dl/work/lsa-jsbiteen/MIGRATED/Lab_Members/Sam_Steen/Data/260217_PAmCherry-Swi6_varied_20ms/chim": "chim",
    # "/home/dl/work/lsa-jsbiteen/MIGRATED/Lab_Members/Sam_Steen/Data/260217_PAmCherry-Swi6_varied_20ms/chim16": "chim16",
    # "/home/dl/work/lsa-jsbiteen/MIGRATED/Lab_Members/Sam_Steen/Data/260604_PAmCherry-Swi6_revchim_20ms": "revchim",
    # "/home/dl/work/lsa-jsbiteen/MIGRATED/Lab_Members/Sam_Steen/Data/260606_PAmCherry-Swi6_revchim_20ms": "revchim",
    # "/home/dl/work/lsa-jsbiteen/MIGRATED/Lab_Members/Sam_Steen/Data/260710_PAmCherry-Swi6_various_20ms/neut": "neut",
    # "/home/dl/work/lsa-jsbiteen/MIGRATED/Lab_Members/Sam_Steen/Data/260710_PAmCherry-Swi6_various_20ms/phos": "phos",
    # "/home/dl/work/lsa-jsbiteen/MIGRATED/Lab_Members/Sam_Steen/Data/260710_PAmCherry-Swi6_various_20ms/scram": "scram",
    # "/home/dl/work/lsa-jsbiteen/MIGRATED/Lab_Members/Sam_Steen/Data/260711_PAmCherry-Swi6_various_20ms/neut": "neut",
    # "/home/dl/work/lsa-jsbiteen/MIGRATED/Lab_Members/Sam_Steen/Data/260711_PAmCherry-Swi6_various_20ms/phos": "phos",
    # "/home/dl/work/lsa-jsbiteen/MIGRATED/Lab_Members/Sam_Steen/Data/260711_PAmCherry-Swi6_various_20ms/scram": "scram"
    # "/home/dl/work/lsa-jsbiteen/MIGRATED/Lab_Members/Sam_Steen/Data/260911_PAmCherry-Swi6_varied_20ms/non-pos": "non-pos",
    "/home/dl/work/lsa-jsbiteen/MIGRATED/Lab_Members/Sam_Steen/Data/260911_PAmCherry-Swi6_varied_20ms/non-neg": "non-neg",
    # "/home/dl/work/lsa-jsbiteen/MIGRATED/Lab_Members/Sam_Steen/Data/260912_PAmCherry-Swi6_varied_20ms/non-pos": "non-pos",
    "/home/dl/work/lsa-jsbiteen/MIGRATED/Lab_Members/Sam_Steen/Data/260912_PAmCherry-Swi6_varied_20ms/non-neg": "non-neg",
    "/home/dl/work/lsa-jsbiteen/MIGRATED/Lab_Members/Sam_Steen/Data/260912_PAmCherry-Swi6_varied_20ms/wt": "wt",
    "/home/dl/work/lsa-jsbiteen/MIGRATED/Lab_Members/Sam_Steen/Data/260918_PAmCherry-Swi6_varied_20ms/non-pos": "non-pos_lowac",
    "/home/dl/work/lsa-jsbiteen/MIGRATED/Lab_Members/Sam_Steen/Data/260918_PAmCherry-Swi6_varied_20ms/swap": "swap",
}

EXCLUDED_FILES = []

# Settings for MSD
MSD_SETTINGS = {'t_int': .02, # Integration time in seconds
                't_delay': 0,  # Define the time delay between frames in seconds
                'min_frames': 4, # Define minimum track length (in frames)
                'max_gap': 2, # Define maximum frame allowed within a track
                'pixel_size_um': 0.049, # Define the pixel size in um
                }

# TODO: maybe merge n_components and criterium?
# Settings for gaussian mixture model fitting (to MSD)
GMM_SETTINGS = {'n_init': 1, # Number of iterations to run
                'n_components': 2, # Number of curves to fit to. Can be an integer or 
                                   # a dictionary (condition: number) or the string "optimize" to find an ideal value.
                'criterium': 'bic', # Criterium for selecting the best model. Can be aic or bic.
                'bootstrap_n': 100, # Number of times to bootstrap data for CI. By far slower than anisotropy stuff-- 100 max
                'verbose': True
                }


# Settings for Anisotropy
ANISOTROPY_SETTINGS = {'min_D': .1, # um^2/sec. Minimum distance to be considered
                       'max_D': 100, # um^2/sec. Maximum distance to be considered
                       'rose_n_bins': 16, # Number of bins to use on the roseplots
                       'angle_from_center': 30, # Degrees in each direction from 0/180° for calculating fold anisotropy
                       'pixel_size_um': MSD_SETTINGS['pixel_size_um'], # Adjust in MSD_SETTINGS
                       'usable_range': (0, .35), # mean displacements to consider for the fold anisotropy graphs
                       'disp_n_bins': 10, # How many mean displacement bins to use on the fold anisotropy graphs
                       'bootstrap_n': 100, # How many bootstrap samples to collect (for error bars on graphs). Often 100-10000.
                       'permutation_n': 100,  # How many permutation samples to collect (for p-values). Often 1000-100000.
                       'min_mean_displacement': .035, # For doing the bar chart comparison only on anisotropies (cont'd next line)
                       'max_mean_displacement': .175 # (cont'd) calculated from steps with a certain range of mean displacements
                      }

In [ ]:
# Initialize combined_tracks (1 row per track) by reading in data from DATA_DIRECTORIES
combined_tracks = fits_convert_trackmate.batch_convert_all_folders(DATA_DIRECTORIES, EXCLUDED_FILES)

# Initialize combined_info (1 row per condition) as a blank data frame
combined_info = pd.DataFrame(data={'condition': list(dict.fromkeys(DATA_DIRECTORIES.values()))}).set_index('condition')

# Initialize general_info for non-condition-specific info. By default, stores all variables in all caps (constants)
general_info = {k: v for k, v in globals().items() if k.isupper() and not k.startswith("__")}

# Save all to file. This is done automatically elsewhere, but manually here since we just created some structures
utils.pickle_save(ct=combined_tracks, ci=combined_info, gi=general_info)

In [4]:
combined_tracks = msd_calc.calc_squared_displacement(combined_tracks, MSD_SETTINGS)

In [5]:
## This cell: calculate MSDs
# Calculate MSD for each track (uses Chris' code, see other file for details)
# combined_tracks, general_info = calc_msd(combined_tracks, general_info, MSD_SETTINGS)
# combined_tracks, general_info = msd_chris(combined_tracks, general_info, MSD_SETTINGS)
combined_tracks, general_info = msd_calc.get_msd(combined_tracks, general_info, MSD_SETTINGS)

In [6]:
# Calculate GMMs and store them as columns (n_comps, means, variances, weights) in combined_info
combined_info = msd_calc.calc_gmms(combined_tracks, combined_info, GMM_SETTINGS)

wt
non-neg
non-pos_lowac
swap


In [7]:
# Calculate angle (for anisotropy) for each track
combined_tracks = calc_anisotropy.calc_angle(combined_tracks, False, ANISOTROPY_SETTINGS)
# Calculate fold anisotropy for each condition
combined_info = calc_anisotropy.calc_fold_anisotropy(combined_info, combined_tracks, ANISOTROPY_SETTINGS)

/home/dl/work/python/python-all-the-way-down/calc_anisotropy.py:96: RuntimeWarning: invalid value encountered in scalar divide
  angle = np.degrees(np.arccos(np.dot(A, B)/((A[0]**2 + A[1]**2)**.5 * (B[0]**2 + B[1]**2)**.5)))
/home/dl/work/python/python-all-the-way-down/calc_anisotropy.py:96: RuntimeWarning: invalid value encountered in arccos
  angle = np.degrees(np.arccos(np.dot(A, B)/((A[0]**2 + A[1]**2)**.5 * (B[0]**2 + B[1]**2)**.5)))


In [8]:
combined_info = calc_anisotropy.get_anisotropy_by_displacement(combined_tracks, combined_info, ANISOTROPY_SETTINGS)

In [9]:
combined_tracks

,index,frame,y,x,track_num,roi_num,condition,file,trajectory,squared_displacement,x_um,y_um,MSD,displacement,angle,mean_displacement
0,0.0,"[713.0, 714.0, 715.0, 716.0, 718.0, 719.0, 720...","[107.91007849428942, 108.52845922662944, 107.1...","[168.2337542157299, 171.16636133862923, 169.72...",0.0,0,wt,/home/dl/work/lsa-jsbiteen/MIGRATED/Lab_Member...,0,"[0.021567172821049585, 0.009719568359037575, 0...","[8.243453956570766, 8.387151705592833, 8.31670...","[5.287593846220182, 5.317894502104843, 5.24892...",NaN,"[0.09495267155300928, 0.1468576617716951, 0.09...","[147.51690293521946, 128.73582162234314, 158.6...","[0.12272276651447593, 0.10368542038511507, 0.1..."
1,297.0,"[14401.0, 14402.0, 14403.0, 14404.0, 14405.0, ...","[90.23236474399884, 88.59334691635905, 88.4778...","[74.71661545050135, 75.99190203060274, 76.6482...",297.0,1,wt,/home/dl/work/lsa-jsbiteen/MIGRATED/Lab_Member...,1,"[0.010354877456999343, 0.0010662904810280966, ...","[3.6611141570745662, 3.7236031994995344, 3.755...","[4.421385872455944, 4.341073998901594, 4.33541...",0.208696,"[0.08941708682812134, 0.10175891831677139, 0.0...","[42.13203103537382, 100.13768204395588, 46.044...","[0.06720651095156074, 0.07207297943367277, 0.1..."
2,296.0,"[14350.0, 14351.0, 14352.0, 14354.0, 14355.0]","[75.37793493125346, 66.548212491645, 62.836092...","[84.77223587401596, 89.06284837150763, 90.4708...",296.0,1,wt,/home/dl/work/lsa-jsbiteen/MIGRATED/Lab_Member...,2,"[0.2313924228679406, 0.03784512715436356, nan,...","[4.153839557826783, 4.364079570203874, 4.43307...","[3.69351881163142, 3.260862412090605, 3.078968...",1.590296,"[0.863903430449167, 0.4810326629948746, 0.1513...",[5.145086037222503],[0.3377854519241735]
3,295.0,"[14264.0, 14265.0, 14267.0, 14268.0, 14269.0]","[75.59757292094314, 72.55462501421297, 74.5709...","[66.3275802201592, 66.96838254111091, 66.73490...",295.0,1,wt,/home/dl/work/lsa-jsbiteen/MIGRATED/Lab_Member...,3,"[0.02321805314584302, nan, 0.01187632632410612...","[3.250051430787801, 3.281450744514435, 3.27001...","[3.704281073126214, 3.555176625696436, 3.65397...",NaN,"[0.10697261144042923, 0.09945912762464805, 0.1...",[133.35289045498766],[0.10415848687880631]
4,294.0,"[13960.0, 13961.0, 13962.0, 13963.0, 13964.0, ...","[69.1494276950014, 69.57808108073449, 75.82451...","[63.02950355805535, 62.97682173227084, 67.4782...",294.0,1,wt,/home/dl/work/lsa-jsbiteen/MIGRATED/Lab_Member...,4,"[0.00044783235878410605, 0.1423338966933951, 0...","[3.0884456743447126, 3.0858642648812715, 3.306...","[3.388321957055069, 3.4093259729559904, 3.7154...",0.926878,"[1.1093944061497787, 0.021162049966487, 0.3772...","[42.784746783746016, 160.270329509067, 64.9007...","[0.19921684918784366, 0.2186962428979694, 0.06..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16764,33.0,"[2231.0, 2232.0, 2233.0, 2235.0, 2236.0, 2237....","[168.8882054912493, 169.00735124388368, 167.34...","[93.7856794138058, 89.9895950794782, 96.913533...",33.0,0,swap,/home/dl/work/lsa-jsbiteen/MIGRATED/Lab_Member...,16764,"[0.03463310921285904, 0.12171109563640868, nan...","[4.595498291276484, 4.409490158894433, 4.74876...","[8.275522069071215, 8.2813602109503, 8.2000895...",1.076784,"[0.7321186464224628, 0.18609972921221365, 0.22...","[168.32671426498047, 126.90585125797085, 3.415...","[0.26748545131106294, 0.24920399966917886, 0.2..."
16765,34.0,"[2252.0, 2253.0, 2254.0, 2255.0, 2256.0]","[164.64318399509744, 161.36757871886658, 159.9...","[95.1718042101514, 92.16956694560284, 93.35465...",34.0,0,swap,/home/dl/work/lsa-jsbiteen/MIGRATED/Lab_Member...,16765,"[0.047402987462475094, 0.008058590868317983, 0...","[4.663418406297418, 4.51630878033454, 4.574378...","[8.067516015759775, 7.907011357224463, 7.83855...",0.049568,"[0.402437490100655, 0.2177222713974742, 0.0897...","[82.81285185401286, 177.18839324592196, 45.550...","[0.15374596294670034, 0.20080399205828078, 0.2..."
16766,35.0,"[2360.0, 2361.0, 2362.0, 2363.0, 2364.0, 2365....","[179.90138477952985, 176.2607483263319

In [10]:
combined_info = calc_anisotropy.summarize_midrange_anisotropy(combined_tracks, combined_info, ANISOTROPY_SETTINGS)

wt vs wt p-val: 1.0
wt vs non-neg p-val: 0.0
wt vs non-pos_lowac p-val: 0.12
wt vs swap p-val: 0.03
